# Random Forest

Binary classification on California housing: `median_house_value > median`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report, roc_curve, auc)

data_df = pd.read_csv("housing.csv")
data_df['total_bedrooms'] = data_df['total_bedrooms'].fillna(data_df['total_bedrooms'].median())
data_df = pd.get_dummies(data_df, columns=['ocean_proximity'], dtype=float)

threshold = data_df['median_house_value'].median()
y = (data_df['median_house_value'] > threshold).astype(int).to_numpy()
X = data_df.drop(columns=['median_house_value']).to_numpy()
feature_names = list(data_df.drop(columns=['median_house_value']).columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, shuffle=True, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)

## Baseline

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print(f"Train acc: {accuracy_score(y_train, rf.predict(X_train)):.4f}")
print(f"Test acc:  {accuracy_score(y_test, y_pred):.4f}")
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=['Below', 'Above']))

## n_estimators sweep

In [ ]:
ns = [10, 25, 50, 100, 200, 400]
train_acc, test_acc = [], []

for n in ns:
    m = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=-1)
    m.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, m.predict(X_train)))
    test_acc.append(accuracy_score(y_test, m.predict(X_test)))
    print(f"n={n:<5} train={train_acc[-1]:.4f} test={test_acc[-1]:.4f}")

plt.figure(figsize=(9, 5))
plt.plot(ns, train_acc, 'o-', label='train')
plt.plot(ns, test_acc, 's-', label='test')
plt.xlabel('n_estimators')
plt.ylabel('accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Depth sweep

In [ ]:
depths = [3, 5, 8, 12, 16, 20, None]
d_train, d_test = [], []

for d in depths:
    m = RandomForestClassifier(n_estimators=200, max_depth=d, random_state=42, n_jobs=-1)
    m.fit(X_train, y_train)
    d_train.append(accuracy_score(y_train, m.predict(X_train)))
    d_test.append(accuracy_score(y_test, m.predict(X_test)))
    print(f"depth={str(d):<5} train={d_train[-1]:.4f} test={d_test[-1]:.4f}")

## Feature importance

In [ ]:
importance = pd.Series(rf.feature_importances_, index=feature_names).sort_values()

plt.figure(figsize=(9, 6))
importance.plot(kind='barh', color='forestgreen')
plt.xlabel('importance')
plt.grid(alpha=0.3)
plt.show()

print(importance)

## OOB score

In [ ]:
rf_oob = RandomForestClassifier(n_estimators=200, oob_score=True,
                                bootstrap=True, random_state=42, n_jobs=-1)
rf_oob.fit(X_train, y_train)
print(f"OOB score: {rf_oob.oob_score_:.4f}")
print(f"Test acc:  {accuracy_score(y_test, rf_oob.predict(X_test)):.4f}")

## ROC

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, label=f'AUC = {roc_auc:.4f}')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.legend()
plt.grid(alpha=0.3)
plt.show()